In [1]:
import torch
import pandas as pd
import numpy as np
import random
from pyfaidx import Fasta

import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from model_v2_compatible import SeqNN

In [2]:
# functions

def one_hot_encode_sequence(sequence_obj):
    # Convert pyfaidx.Sequence object to string
    sequence = str(sequence_obj).upper()
    
    # Define the mapping from bases to integers
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    valid_bases = list(base_to_int.keys())

    # Step 1: Convert sequence to integer encoding with random base for 'N'
    encoded_indices = []
    for base in sequence:
        if base in base_to_int:
            encoded_indices.append(base_to_int[base])
        else:
            random_base = random.choice(valid_bases)
            encoded_indices.append(base_to_int[random_base])

    # Step 2: One-hot encode the sequence
    encoded_sequence = np.array(encoded_indices)
    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    input_sequence = np.expand_dims(one_hot_encoded, axis=0)

    return input_sequence


def upper_triangular_to_vector_skip_two_diagonals(matrix):
    """
    Extracts the upper triangular part of a square matrix (excluding the first two diagonals) 
    and transforms it into a vector.
    
    Parameters:
        matrix (np.ndarray): A 2D numpy matrix of shape (512, 512).
        
    Returns:
        np.ndarray: A 1D array containing the upper triangular elements (excluding the first two diagonals).
    """
    if matrix.shape != (512, 512):
        raise ValueError("Input matrix must be of shape (512, 512).")
    
    # Extract the upper triangular part excluding the first two diagonals
    upper_triangular_vector = matrix[np.triu_indices(512, k=2)]
    
    return upper_triangular_vector


def fragment_indices_in_upper_triangular(matrix_size=512, fragment_mask=None):
    """
    Given a binary mask for a fragment in a (448, 448) matrix, find the corresponding indices 
    within the upper triangular output vector (excluding the first two diagonals).

    Parameters:
        matrix_size (int): The size of the square matrix (default: 448).
        fragment_mask (np.ndarray): A boolean mask of shape (448, 448) marking the fragment.

    Returns:
        np.ndarray: Indices in the upper triangular vector corresponding to the fragment.
    """
    if fragment_mask.shape != (matrix_size, matrix_size):
        raise ValueError("Fragment mask must be of shape (448, 448).")

    # Get the upper triangular indices skipping two diagonals
    row_indices, col_indices = np.triu_indices(matrix_size, k=2)
    
    # Identify which of these indices are in the fragment
    selected_indices = np.where(fragment_mask[row_indices, col_indices])[0]
    
    return selected_indices


def store_tower_output(ohe_sequence, model, path):
    x = model.conv_block_1(ohe_sequence)
    x = model.conv_tower(x)
    # save the tensor
    torch.save(x, path)
    torch.cuda.empty_cache()

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

/tmp/SLURM_1428979/ipykernel_1624276/2744654124.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [5]:
FOLD = 2

In [6]:
df = pd.read_csv(f"/scratch1/smaruj/genomic_flat_regions/flat_regions_chrom_states_tsv/fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv", sep="\t")

In [7]:
fasta_file = "/project/fudenber_735/genomes/mm10/mm10.fa"
genome = Fasta(fasta_file)

In [8]:
bin_size = 2048
cropping_applied = 64
padding_bins = 0
padding = padding_bins * bin_size

slice_0_bins = [256]
slice_0_start = (min(slice_0_bins) + cropping_applied - padding_bins) * bin_size
slice_0_end = (max(slice_0_bins) + 1 + cropping_applied + padding_bins) * bin_size

In [9]:
slice_0_start, slice_0_end

(655360, 657408)

In [10]:
c = -0.5

In [11]:
CTCF_PWM = "/home1/smaruj/IterativeMutagenesis/MA0139.1.meme"

In [12]:
def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list)
    
    return pwm_array

In [13]:
pwm_CTCF = read_meme_pwm_as_numpy(CTCF_PWM)
pwm_CTCF_tensor = torch.from_numpy(pwm_CTCF.T).float()
motifs_dict = {"CTCF": pwm_CTCF_tensor}

In [14]:
from tangermeme.tools import fimo

In [15]:
ctcf_scan_flank = 20

In [16]:
ctcf_locations_list = []

for row in df.itertuples(index=False):
    chrom, pred_start, pred_end = row.chrom, row.centered_start, row.centered_end
    sequence = genome[row.chrom][pred_start:pred_end]
    
    X = one_hot_encode_sequence(sequence)
    X_tensor = torch.tensor(X)
    
    # modifying X
    X_mod = X_tensor.clone()
    
    mod_slice = torch.load(f"/scratch1/smaruj/generate_genomic_boundary/results/target_-0.5/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_slice.pt", map_location=device)
    
    X_mod[0, :, slice_0_start:slice_0_end] = mod_slice[0, :, :]
        
    slice_to_scan = X_mod[:, :, slice_0_start-ctcf_scan_flank:slice_0_end+ctcf_scan_flank]
    
    hits = fimo.fimo(
        motifs=motifs_dict,
        sequences=slice_to_scan,
        threshold=1e-4,
        reverse_complement=True
    )[0]
    
    hits["start"] -= ctcf_scan_flank
    hits["end"] -= ctcf_scan_flank
    
    # Store as list of tuples
    locs = [
        (int(s), int(e))
        for s, e in zip(hits["start"], hits["end"])
    ]
    ctcf_locations_list.append(locs)

/tmp/SLURM_1428979/ipykernel_1624276/1443168775.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mod_slice = torch.load(f"/scratch1/smaruj/generate_genomic_boundary/resu

In [17]:
# Add column of lists to your DataFrame
df["ctcf_motif_locs"] = ctcf_locations_list

In [18]:
df

,chrom,fold,PearsonR,centered_start,centered_end,centered_flat_start,centered_flat_end,active_fraction,neutral_fraction,repressive_fraction,ctcf_motif_locs
0,chr11,fold2,0.732358,45981696,47292416,194,318,0.222222,0.555556,0.222222,"[(830, 849), (1596, 1615), (1686, 1705), (1749..."
1,chr11,fold2,0.767461,56039424,57350144,173,339,0.440000,0.520000,0.040000,"[(784, 803), (1384, 1403), (245, 264), (337, 3..."
2,chr11,fold2,0.791830,39716864,41027584,192,320,0.444444,0.555556,0.000000,[]
3,chr11,fold2,0.833249,42188800,43499520,171,341,0.466667,0.533333,0.000000,"[(195, 214), (249, 268), (1932, 1951)]"
4,chr11,fold2,0.836212,44619776,45930496,170,342,0.428571,0.523810,0.047619,"[(144, 163), (922, 941), (1008, 1027)]"
5,chr11,fold2,0.910790,55146496,56457216,190,322,0.411765,0.529412,0.058824,"[(1355, 1374), (1426, 1445), (176, 195), (1343..."
6,chr11,fold2,0.915895,43421696,44732416,160,352,0.411765,0.529412,0.058824,"[(1109, 1128), (140, 159), (689, 708), (761, 7..."
7,chr14,fold2,0.731114,10399744,11710464,168,344,0.382353,0.500000,0.117647,"[(1106, 1125)]"
8,chr14,fold2,0.738000,11220992,12531712,187,325,0.157895,0.526316,0.315789,"[(306, 325), (394, 413), (1731, 1750)]"
9,chr14,fold2,0.763067,17168384,18479104,204,308,0.428571,0.571429,0.000000,"[(1810, 1829), (1956, 1975), (541, 560), (634,..."


In [19]:
df.to_csv(f"/scratch1/smaruj/suppressing_CTCFs/fold{FOLD}_with_positions.tsv", sep="\t", index=False)

In [20]:
for row in df.itertuples(index=False):
    chrom, pred_start, pred_end = row.chrom, row.centered_start, row.centered_end
    sequence = genome[row.chrom][pred_start:pred_end]
    
    X = one_hot_encode_sequence(sequence)
    X_tensor = torch.tensor(X)
    
    model.eval()
    with torch.no_grad():
        y = model(X_tensor)
    
    torch.save(y, f"/scratch1/smaruj/suppressing_CTCFs/targets/target_{c}/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_target.pt")
    
    # modifying X
    X_mod = X_tensor.clone()
    
    mod_slice = torch.load(f"/scratch1/smaruj/generate_genomic_boundary/results/target_{c}/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_slice.pt", map_location=device)
    
    X_mod[0, :, slice_0_start:slice_0_end] = mod_slice[0, :, :]
        
    torch.save(X_mod, f"/scratch1/smaruj/suppressing_CTCFs/ohe_X/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_X.pt")

    store_tower_output(X_mod, model, f"/scratch1/smaruj/suppressing_CTCFs/tower_outputs/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_tower_out.pt")

/tmp/SLURM_1428979/ipykernel_1624276/1556192372.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mod_slice = torch.load(f"/scratch1/smaruj/generate_genomic_boundary/resu